In [0]:
import os 
import pandas as pd
# from sklearn.preprocessing import RobustScaler
import numpy as np
import ast
from pathlib import Path


In [0]:
# Get the current directory
current_directory = os.getcwd()
parent_directory = Path(current_directory).parent

# Get the source file path
file_path = f'{parent_directory}/datasets/raw'
file = f"{file_path}/labelled_training_data.csv"
df = pd.read_csv(file, header=0)
df

![process_dataset_data_description.png](attachment:process_dataset_data_description.png)


In [0]:
from parse_argument_v2 import *
df = preprocess_args_column(df, col="args", output_col="argsText")
print(df["argsText"].head())


In [0]:
df["EventText"] = df["eventName"].astype(str) + " " + df["argsText"].astype(str)
df.to_csv(f"./parsed_arguments_training_data.csv", index=False)

#### 1. Prepare process and user identifiers columns

In [0]:
df.columns

In [0]:
# todo: convert ['processId','threadId','parentProcessId','userId','mountNamespace','eventId'] to categorical features

# threadId: While this value did not appear useful in our
# analysis, it might suggest how to link process calls if obfus-
# cated in the system. No conversion is recommended at this
# time.

# parentProcessId: Same as processId, the same
# mapping to a binary variable should suffice.
df['is_parent_system_process'] = df['parentProcessId'].isin([0, 1, 2]).astype(int)

# processId: 
    # Process IDs 0, 1, and 2 are meaningful since
    # these are always values used by the OS, but otherwise a
    # random number is assigned to the process upon creation. We
    # recommend replacing processId with a binary variable
    # indicating whether or not processID is 0, 1, or 2.

df['is_system_process'] = df['processId'].isin([0, 1, 2]).astype(int)

In [0]:
# get parent process info and append it to the dataframe
df_parent_process = df[["hostName","processId","processName","userId"]].drop_duplicates()
df = df.join(df_parent_process.set_index(["hostName","processId"]), on=["hostName","parentProcessId"], rsuffix="_parent")
df = df.rename(columns={"processName_parent": "parentProcessName", "userId_parent": "parentUserId"})

In [0]:
# What: Binary encoding of userId based on whether it is below 1000 or not.
# Why: Distinguish between system/OS users and regular users, as system activities often
df["userId_binary"]  = df["userId"].apply(lambda x: 1 if x < 1000 else 0)

# What: Did the child process run under the same user as its parent?
# Why: Malicious processes may run under different user accounts than their parent processes.
df["same_user_as_parent"] = np.where(df["parentUserId"].notnull(),(df["userId"] == df["parentUserId"]).astype(int), -1)

    

# What: Did the parent fork and re-exec itself or spawn a different binary?
# why: Malicious activity may involve a process spawning a different binary than itself.
df["same_process_name_as_parent"] = np.where(df["parentProcessName"].notnull(),(df["processName"] == df["parentProcessName"]).astype(int), -1)




In [0]:
# Adding frequency features for multiple columns
# what: Frequency encoding for multiple columns to capture commonality within each host.
for col in ['processId','threadId','parentProcessId','userId','mountNamespace','eventId']:
    df[col + "_freq"] = df.groupby(["hostName", col])[col].transform("size")

# exam the data
df[["hostName", "processId", "processId_freq"]].sort_values(by=["hostName", "processId"]).drop_duplicates()

#### 2. Transform stackAddresses column to numeric features

In [0]:
# convert stackAddresses from string to list of integers
df['stackAddresses'] = df['stackAddresses'].apply(lambda x: [int(i) for i in x.strip('[]').strip(' ').split(',') if i])


In [0]:
# calculate the length of a stackAddresses
df['stackAddresses_len'] = df['stackAddresses'].apply(len)
df['stackAddresses_len'].describe()

In [0]:
# normal stacks are often continuous, while abnormal stacks may have large jumps.
# calculate the standard deviation of the differences between consecutive stack addresses
def stack_jump_std(stack):
    if len(stack) < 2:
        return 0
    diffs = [abs(stack[i] - stack[i+1]) for i in range(len(stack)-1)]
    return np.std(diffs)
df['stackAddresses_jump_std'] = df['stackAddresses'].apply(stack_jump_std)
df['stackAddresses_jump_std'].describe()

In [0]:
# For normal function call, the stackAddresses should be highly diverse.
# malicious or abnormal behavior, it often manipulates and uses stack addresses.
df['stackAddresses_unique_ratio'] = df['stackAddresses'].apply(lambda x: len(set(x)) / max(len(x), 1))
df['stackAddresses_unique_ratio'].describe()

#### 3. add a new feature indicating whether a process end with an error.

In [0]:
df["returnValue_is_error"] = (df["returnValue"] == -1).astype(int)

#### 4. TODO: process arg column.

In [0]:
# args: There are many options in this variable list of dictio-
# naries. For simplicity, we refrain from utilising any of these
# values. However, more features can and should be created
# for future work.

def argument_parse(args_str):
    return ast.literal_eval(args_str)

df['args'] = df['args'].apply(argument_parse)

df['args_has_path'] = df['args'].apply(lambda x: int(any('pathname' in d['name'] for d in x)))
df[['args_has_path', 'args']].head(10)

#### 4. process mountNamespace column.

In [0]:
# mountNamespace: This field is somewhat consistent
# across our hosts and determines the access a certain pro-
# cess has to various mount points. The most common value
# for this feature is 4026531840 or 0xF0000000, which is
# for the mnt/ directory where all manually mounted points
# are linked. It is noted that all logs with userId ≥1000
# had a mountNamespace of 4026531840, while some OS
# userId traffic used different mountNamespace values.
# We converted this feature into a binary mappin

def mount_ns_binary(ns):
    return int(ns == 4026531840)

df['mountNamespace_binary'] = df['mountNamespace'].apply(mount_ns_binary)

#### 5. scale the numeric features

In [0]:
# scaler = RobustScaler()
# df[["stackAddresses_len", "stackAddresses_jump_std","argsNum"]] = scaler.fit_transform(df[["stackAddresses_len", "stackAddresses_jump_std","argsNum"]])
# df.head()


In [0]:
df.columns

In [0]:
folder_path = Path(f"{current_directory}/datasets/processed/")

# Create the directory(ies).
# parents=True ensures missing parent directories are created.
# exist_ok=True prevents an error if the directory already exists.
folder_path.mkdir(parents=True, exist_ok=True)

print(f"Folder structure '{folder_path}' created successfully.")

In [0]:
df[['sus', 
    'evil',
    'eventId',
    'is_system_process',
    'processId_freq', 
    'threadId_freq', 
    'parentProcessId_freq',
    'userId_freq', 
    'mountNamespace_freq', 
    'eventId_freq', 
    'userId_binary',
    'same_user_as_parent',
    'same_process_name_as_parent',
    'stackAddresses_len', 
    'stackAddresses_jump_std',
    'stackAddresses_unique_ratio', 
    'returnValue',
    'returnValue_is_error', 
    'argsNum',
    'args_has_path',
    'mountNamespace_binary']].to_csv(f"{current_directory}/datasets/processed/processed_training_data.csv", index=False)